In [ ]:
'''
Notebook to prototype metropolis hasings sampling of soft spheres

Copyright (C) 2026 Miriam Derla

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/>
'''

In the following we perform a metropolis-hastings sampling of the canonical ensemble for a system of *penetrable spheres* (PS) and *penetrable disks* (PD). The "penetrable" is meant in the sense that they interact via the pair potential $u_E(r)=E\,\Theta(\sigma-r)$. This treatment of penetrable spheres/disks can be understood as a variation on the problem of predicting the free energy of a system of *rigid spheres* (RS) or *rigid disks* (RD), also referrred to as hard spheres or hard disks: when $E\gg k_\text{B}T$, a pair of particles would need an unrealistically high energy to overcome the barrier $E$ to closing in to any distance shorter than $\sigma$, effectively making them rigid. The family $\{u_E\}_{E\in[0,\infty)}$ in this sense connects the ideal gas, interacting with $u_0(r)\equiv 0$, to the rigid particle fluid, doing so with potentials that have a clear mathematical resemblance with rigid particle interaction.

The *overlap energy* $E$ (or *penetration energy barrier*) is the only energy scale in the system besides temperature, such that the phase diagram only depends on the ratio $$\begin{equation}\varepsilon=\frac{E}{k_\text{B}T}\end{equation}$$ Besides the reduced overlap energy $\varepsilon$, the system behaviour is still shaped by the *packing density* $$\begin{equation}\Phi=B_d\left(\frac{\sigma}{2}\right)^d\rho\end{equation}$$ where $\rho$ is number density and $B_d$ the volume of a unit sphere in $d$ dimensions (which reads, forexample, $B_2=\pi$ and $B_3=\frac{4}{3}\pi$ in $d=3$)

In [ ]:
import numpy as np
import numba as nb

In [ ]:
from math import lgamma

@nb.njit
def B(d):
    '''-
    Returns the hyper-volume of the unit ball
    in d dimensions

    https://en.wikipedia.org/wiki/N-sphere#Volume_and_area
    '''
    gamma_of_1_plus_d_over_2 = np.exp(lgamma(1 + d/2))
    return np.pi**(d/2) / gamma_of_1_plus_d_over_2

In [ ]:
@nb.njit
def u(r, E, sigma):
	'''
	pair potential for penetrable spheres or disks given a radius
	'''
	return E if r < sigma else (E/2 if r == sigma else 0)

In [ ]:
from random import shuffle


@nb.njit
def run(
    overlap_energy, kB_times_temperature,
    packing_density, particle_number, spatial_dimension_count,
    simulation_box_size=1, move_count=10000):
    '''
    Executing the metropolis hastings sampling of
    the penetrable sphere system
    '''
    # computing the particle diameter corresponding to the ...
    # ... sought packing density
    volume = simulation_box_size**spatial_dimension_count
    number_density = particle_number / volume
    sigma = 2 * (
        packing_density / (
            B(spatial_dimension_count) * number_density)
            )**(1/spatial_dimension_count)
    # randomly placed particles
    x = np.random.rand(particle_number, spatial_dimension_count)
    # cell list for energy computation
    # TODO DO
    # executing the sampling moves ...
    for move in range(move_count):
		# ... where we update non-interacting cells in parallel ...
        # ... according to a checkerboard pattern
        u(0, overlap_energy, sigma)
        # shuffling all particles
        shuffle(x)


In [ ]:
run(1, 1, 0.3, 100, 3)